In [ ]:
import ee
import geemap
import numpy as np
import calendar

In [ ]:
# .env vars
project_id='musa-wildfire-449918'
asset_path = 'projects/musa-wildfire-449918/assets/angola_eco-regions_30m'
file_base_name = 'angola'
scale = 1000

In [ ]:
def initialize_earth_engine(project_id):
    ee.Authenticate()
    ee.Initialize(project=project_id)
    print(f"Earth Engine initialized with project: {project_id}")
    return project_id

In [ ]:
def get_dry_months(area, start_year, end_year, analysis_year):
    # Load TerraClimate dataset
    dataset = ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE') \
        .filter(ee.Filter.calendarRange(start_year, end_year, 'year')) \
        .select('pdsi')

    monthly_avg_pdsi = []
    for month in range(1, 13):
        month_collection = dataset.filter(ee.Filter.calendarRange(month, month, 'month'))

        # Compute mean only where the ecoregion mask is nonzero
        month_mean = month_collection.mean().clip(area) \
            .reduceRegion(
                reducer=ee.Reducer.mean(),
                scale=4000,  # Match raster resolution
                bestEffort=True,
                geometry = area.geometry()
            ).get('pdsi')  # Ensure 'pdsi' is the correct band name

        monthly_avg_pdsi.append(month_mean)

    # Convert monthly_avg_pdsi into an Earth Engine ee.List if it's a regular Python list
    monthly_avg_pdsi_ee = ee.List(monthly_avg_pdsi)

    # Compute the rolling sums for 3-month periods
    rolling_sums = ee.List([
        monthly_avg_pdsi_ee.slice(i, i + 3).reduce(ee.Reducer.sum())
        for i in range(10)  # 12 months → 10 possible 3-month windows
    ])

    # Find the minimum rolling sum value
    min_value = rolling_sums.reduce(ee.Reducer.min())

    # Get the index of the minimum rolling sum
    min_index = rolling_sums.indexOf(min_value)

    start_month_number = min_index.add(1)  # 1-based index for month numbers
    mid_month_number = min_index.add(2)   # 1-based index for the next month
    end_month_number = min_index.add(3)   # 1-based index for the third month

    start_month_number = start_month_number.mod(12)
    mid_month_number = mid_month_number.mod(12)
    end_month_number = end_month_number.mod(12)

    # Store them as a list
    dry_months = [
        start_month_number.getInfo(),
        mid_month_number.getInfo(),
        end_month_number.getInfo()
    ]

   # Convert index to month names
    months = ee.List(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
    start_month = months.get(min_index)
    mid_month = months.get(min_index.add(1))
    end_month = months.get(min_index.add(2))

    # Output the driest 3-month period
    print('Driest 3-month period:', start_month.getInfo(), mid_month.getInfo(), end_month.getInfo())

    last_dry_day = calendar.monthrange(analysis_year, dry_months[0])[1]
    start_date = str(analysis_year) + '-' + str(dry_months[0]).zfill(2) + '-01'
    end_date = str(analysis_year) + '-' + str(dry_months[2]).zfill(2) + '-' + str(last_dry_day).zfill(2)

    return start_date, end_date, dry_months

In [ ]:
def get_fire_season_months(area, start_year, end_year, analysis_year):

    burned_collection = ee.ImageCollection("MODIS/061/MCD64A1") \
        .filter(ee.Filter.calendarRange(start_year, end_year, 'year')) \
        .select('BurnDate')

    base_date = ee.Date.fromYMD(analysis_year, 1, 1)

    month_burns = []

    for m in range(1, 13):
        start = ee.Date.fromYMD(analysis_year, m, 1)
        end = start.advance(1, 'month')

        start_doy = start.difference(base_date, 'day')
        end_doy = end.difference(base_date, 'day')

        monthly_img = burned_collection.map(
            lambda img: img.gte(start_doy)
                        .And(img.lt(end_doy))
                        .And(img.gt(0))
                        .selfMask()
        ).sum()

        monthly_total = monthly_img.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=area,
            scale=500,
            bestEffort=True
        ).get('BurnDate')

        month_burns.append(monthly_total)

    # Compute rolling 3-month sums
    rolling_sums = []
    for i in range(10):  # 12 months → 10 windows
        sum_3 = ee.Number(month_burns[i]) \
                    .add(ee.Number(month_burns[i+1])) \
                    .add(ee.Number(month_burns[i+2]))
        rolling_sums.append(sum_3)

    # Find the index of the max rolling sum
    max_val = ee.List(rolling_sums).reduce(ee.Reducer.max())
    max_idx = ee.List(rolling_sums).indexOf(max_val)

    # Convert to 1-based month numbers
    start_month = max_idx.add(1)
    mid_month = max_idx.add(2)
    end_month = max_idx.add(3)

    # Get values client-side
    dry_months = [start_month.getInfo(), mid_month.getInfo(), end_month.getInfo()]
    last_day = calendar.monthrange(analysis_year, dry_months[2])[1]

    start_date = f"{analysis_year}-{str(dry_months[0]).zfill(2)}-01"
    end_date = f"{analysis_year}-{str(dry_months[2]).zfill(2)}-{str(last_day).zfill(2)}"

    return start_date, end_date, dry_months


In [ ]:
import ee
import calendar

def get_fire_season_months(area, start_year, end_year, analysis_year, season_length):

    burned_collection = ee.ImageCollection("MODIS/061/MCD64A1") \
        .filter(ee.Filter.calendarRange(start_year, end_year, 'year')) \
        .select('BurnDate')

    base_date = ee.Date.fromYMD(analysis_year, 1, 1)

    month_burns = []

    for m in range(1, 13):
        start = ee.Date.fromYMD(analysis_year, m, 1)
        end = start.advance(1, 'month')

        start_doy = start.difference(base_date, 'day')
        end_doy = end.difference(base_date, 'day')

        monthly_img = burned_collection.map(
            lambda img: img.gte(start_doy)
                        .And(img.lt(end_doy))
                        .And(img.gt(0))
                        .selfMask()
        ).sum()

        monthly_total = monthly_img.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=area,
            scale=500,
            bestEffort=True
        ).get('BurnDate')

        month_burns.append(monthly_total)

    # Compute rolling season_length sums
    rolling_sums = []
    for i in range(12 - season_length + 1):  # 12 months → (12 - season_length + 1) windows
        sum_season = ee.Number(0)
        for j in range(season_length):
            sum_season = sum_season.add(ee.Number(month_burns[i + j]))
        rolling_sums.append(sum_season)

    # Find the index of the max rolling sum
    max_val = ee.List(rolling_sums).reduce(ee.Reducer.max())
    max_idx = ee.List(rolling_sums).indexOf(max_val)

    # Convert to 1-based month numbers
    dry_months = [max_idx.add(i).add(1).getInfo() for i in range(season_length)]

    # Get the last day of the last month in the dry season
    last_day = calendar.monthrange(analysis_year, dry_months[-1])[1]

    start_date = f"{analysis_year}-{str(dry_months[0]).zfill(2)}-01"
    end_date = f"{analysis_year}-{str(dry_months[len(dry_months)-1]).zfill(2)}-{str(last_day).zfill(2)}"

    return start_date, end_date, dry_months

In [ ]:
def get_wet_months(dry_months, analysis_year):
    """
    Shifts a list of three consecutive months backwards by three months.
    Handles cases where shifting crosses into the previous year.

    Args:
        dry_months (list of int): List of three month numbers (1-12).

    Returns:
        tuple: (list of int, bool) - The three previous months and a flag
               indicating if they belong to the previous year.
    """
    shifted_months = [(month - 3) if month > 3 else (month - 3 + 12) for month in dry_months]

    last_wet_day = calendar.monthrange(analysis_year, shifted_months[2])[1]

    # If any of the shifted months is greater than the corresponding original month,
    # it means we wrapped around to the previous year.
    previous_year = any(shifted_months[i] > dry_months[i] for i in range(3))
    if previous_year:
        analysis_year -= 1
    time_shift_start_date = str(analysis_year) + '-' + str(shifted_months[0]).zfill(2) + '-01'
    time_shift_end_date = str(analysis_year) + '-' + str(shifted_months[2]).zfill(2) + '-' + str(last_wet_day).zfill(2)


    return time_shift_start_date, time_shift_end_date

In [ ]:
def read_and_clip(id, area, band, start= None, end= None):
  if start is not None and end is not None:
    band = ee.ImageCollection(id) \
    .filter(ee.Filter.date(start, end)) \
    .select(band) \
    .mean() \
    .clip(area)
  else: # for layers that don't have a date range like AGB
    band = ee.ImageCollection(id) \
        .select(band) \
        .mean() \
        .clip(area)
  return band

In [ ]:
def export_raster_to_asset(raster, area, project_id, asset_name, scale):
    """
    Export a raster image to a Google Earth Engine asset.

    Parameters:
    - raster: The raster to export.
    - area: The study area
    - project_id: The Google Earth Engine project ID where the asset will be stored.
    - asset_name: The name of the asset (e.g., "filtered_ecoregions_raster_30m").
    - scale: The scale/resolution of the output raster (in meters per pixel).

    Returns:
    - Export task object.
    """
    area = area.geometry()
    # Define the export task
    export_task = ee.batch.Export.image.toAsset(
        image=raster,
        description=f'Export_{asset_name}',
        assetId=f'projects/{project_id}/assets/{asset_name}',
        region=area,  # Define region of interest as the geometry
        scale=scale,
        maxPixels=1e13,  # Adjust depending on your raster size
    )

    # Start the export task
    export_task.start()

    # Return the export task object for monitoring
    return export_task

In [ ]:
def rasterize_ecoregions(ecoregions, scale=30):
    """Rasterize ecoregion features to a raster image."""
    raster = ecoregions.reduceToImage(
        properties=['ECO_ID'],
        reducer=ee.Reducer.first()
    ).reproject(crs='EPSG:4326', scale=scale)
    return raster

In [ ]:
bands_to_export = [
    {"code": "IDAHO_EPSCOR/TERRACLIMATE", "bands": ["pdsi", "tmmx", "vs", "soil", "pr"], "time": True},
    {"code": "NASA/ORNL/biomass_carbon_density/v1", "bands": ["agb"], "time": False},
    {"code": "projects/musa-wildfire-449918/assets/rasterized_ecoregions_full_30m_unclipped", "bands": [], "time": False},
    {"code": "NASA/NASADEM_HGT/001", "bands": ["elevation", "swb"], "time": False},
    {"code": "MODIS/061/MCD64A1", "bands": ["BurnDate"], "time": True},
]


In [ ]:
initialize_earth_engine(project_id)
asset_data = ee.FeatureCollection(asset_path)

study_area = asset_data
start_date, end_date, dry_months = get_fire_season_months(study_area, 2014, 2024, 2020,3)

#start_date2, end_date2, dry_months2 = get_fire_season_months2(study_area, 2014, 2024, 2020, 4)

Earth Engine initialized with project: musa-wildfire-449918


In [ ]:
print("orig vers")
print(start_date)
print(end_date)
print(dry_months)
print("new vers")


orig vers
2020-06-01
2020-08-31
[6, 7, 8]
new vers


In [ ]:
# get study area and dates of interest
# initialize_earth_engine(project_id)
# asset_data = ee.FeatureCollection(asset_path)

# study_area = asset_data
# start_date, end_date, dry_months = get_fire_season_months(study_area, 2014, 2024, 2020)
# start_time_shift, end_time_shift = get_wet_months(dry_months, 2020)



In [ ]:
# print(start_date)
# print(end_date)
# print(dry_months)
# print(start_time_shift)
# print(end_time_shift)

In [ ]:
study_area_img = rasterize_ecoregions(study_area)

In [ ]:
# Set your target projection (CRS) and scale (30m)
target_crs = study_area_img.projection().crs()  # Or set explicitly like 'EPSG:32618' for UTM zones
target_scale = 30  # meters

# Start with your base image
multi_band_raster = study_area_img
multi_band_raster = multi_band_raster.select(['first']).rename(['eco-regions'])

# Go through all datasets
for dataset in bands_to_export:
    if dataset["time"]:
        print(f"Processing time-dependent dataset: {dataset['code']}")
        for band in dataset["bands"]:
            print(f"  - Band: {band}")
            lyr = read_and_clip(dataset['code'], study_area, band, start_date, end_date)

            # Resample + reproject to target CRS and resolution
            newProj = lyr.projection().atScale(30)
            lyr = lyr.resample('bilinear').reproject(newProj)
            proj = lyr.projection().getInfo()

            transform = proj['transform']

            # Pixel size
            pixel_width = abs(transform[0])
            pixel_height = abs(transform[4])

            print(f"Resolution: {pixel_width} x {pixel_height} degrees or meters (depends on CRS)")
            print(f"CRS: {proj['crs']}")
            print(lyr.projection().getInfo())

            # Add to the stack
            multi_band_raster = multi_band_raster.addBands([lyr])

# Add DEM
dem = ee.Image('NASA/NASADEM_HGT/001').select('elevation').updateMask(study_area_img)

# Resample + reproject DEM
dem = dem.resample('bilinear').reproject(crs=target_crs, scale=target_scale)

multi_band_raster = multi_band_raster.addBands([dem])

# Add above-ground biomass (AGB)
agb = read_and_clip('NASA/ORNL/biomass_carbon_density/v1', study_area, 'agb')

# Resample + reproject AGB
agb = agb.resample('bilinear').reproject(crs=target_crs, scale=target_scale)

multi_band_raster = multi_band_raster.addBands([agb])


Processing time-dependent dataset: IDAHO_EPSCOR/TERRACLIMATE
  - Band: pdsi
Resolution: 0.00026949458523585647 x 0.00026949458523585647 degrees or meters (depends on CRS)
CRS: EPSG:4326
{'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [0.00026949458523585647, 0, 0, 0, 0.00026949458523585647, 0]}
  - Band: tmmx
Resolution: 0.00026949458523585647 x 0.00026949458523585647 degrees or meters (depends on CRS)
CRS: EPSG:4326
{'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [0.00026949458523585647, 0, 0, 0, 0.00026949458523585647, 0]}
  - Band: vs
Resolution: 0.00026949458523585647 x 0.00026949458523585647 degrees or meters (depends on CRS)
CRS: EPSG:4326
{'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [0.00026949458523585647, 0, 0, 0, 0.00026949458523585647, 0]}
  - Band: soil
Resolution: 0.00026949458523585647 x 0.00026949458523585647 degrees or meters (depends on CRS)
CRS: EPSG:4326
{'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [0.00026949458523585647, 0, 0, 

In [ ]:
# multi_band_raster = study_area_img
# multi_band_raster = multi_band_raster.select(['first']).rename(['eco-regions'])
# for dataset in bands_to_export:
#     if dataset["time"]:
#         print(f"Processing time-dependent dataset: {dataset['code']}")
#         for band in dataset["bands"]:
#             print(f"  - Band: {band}")
#             lyr = read_and_clip(dataset['code'], study_area, band, start_date, end_date)
#             multi_band_raster = multi_band_raster.addBands([lyr])
# dem = ee.Image('NASA/NASADEM_HGT/001').select('elevation').updateMask(study_area_img)
# multi_band_raster = multi_band_raster.addBands([dem])
# agb = read_and_clip('NASA/ORNL/biomass_carbon_density/v1', study_area, 'agb')
# multi_band_raster = multi_band_raster.addBands([agb])

# print("MBR Contains")
# print(multi_band_raster.bandNames().getInfo())


In [ ]:
# visualize

# Create map
Map = geemap.Map()

# Get band names
band_names = multi_band_raster.bandNames().getInfo()

# Loop through bands and add to map
for band in band_names:
    band_img = multi_band_raster.select(band)

    # You can customize vis_params per band or keep generic
    vis_params = {"min": 0, "max": 1, "palette": ["white", "blue", "green", "red"]}
    Map.addLayer(band_img, vis_params, band)

# Add layer control and show the map
Map.addLayerControl()
Map


Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [ ]:
Map2 = geemap.Map()

Map2.addLayer(multi_band_raster.select("BurnDate"), {}, "Burn Date")
Map2.addLayer(multi_band_raster.select("soil"), {}, "soil")

Map2

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [ ]:
# export:
file_name = f"{file_base_name}_training_30m_2020_resampled_bilinear_new"
export_raster_to_asset(multi_band_raster, study_area, project_id, file_name, scale)

<Task U5FKGBDHV7HKAPKRPHJSLRU3 Type.EXPORT_IMAGE: Export_angola_training_30m_2020_resampled_bilinear_new (State.UNSUBMITTED)>